In [1]:
from pathlib import Path
import pickle
import itertools
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import eda
import eda_distance

In [2]:
stimuli_dir = Path("card_game/stimuli")  
csv_path = stimuli_dir / f"civ_items_trial_8.csv"
df = pd.read_csv(csv_path)
numeric = df.drop(columns=["Name"], errors="ignore")
items = numeric.to_numpy(dtype=np.int64)

In [ ]:
# unbiased eda run

n_items = items.shape[0]
n_obj = items.shape[1] - 1
n_con = 1

if n_obj == 3:
    n_selected = 6
    max_row_diff = 5 
elif n_obj == 5:
    n_selected = 10
    max_row_diff = 500 
else:
    raise ValueError(f"Number of objectives {n_obj} not supported")

capacity = n_selected * 10
pop_size = 1_000
generations = 100 
max_no_improve_gen = 5
p_rank = None 

eda_process = eda.KnapsackEDA(
    items=items,
    capacity=capacity,
    n_selected=n_selected,
    n_obj=n_obj,
    pop_size=pop_size,
    generations=generations,
    max_no_improve_gen=max_no_improve_gen,
    max_row_diff=max_row_diff,
    seed=1125,
    p_rank=p_rank
)
results = eda_process.run()

if p_rank is not None:
    result_type = "eda_human"
else:
    result_type = "eda"

output_dir = "card_game/eda_results/test"
os.makedirs(output_dir, exist_ok=True)
file_path = os.path.join(output_dir, f"{result_type}_test.pkl")
if os.path.exists(file_path):
    print(f"File {file_path} already exists")
else:
    with open(file_path, 'wb') as f:
        pickle.dump(results, f)

In [4]:
# specify aspiration / check real solution closest to aspi

with open("card_game/eda_results/eda_trial8.pkl", 'rb') as f:
    results_actual = pickle.load(f)
pf_actual = results_actual["converged_pf_table"][-1]

# target = np.median(pf_actual, axis=0)
# qmax = np.percentile(pf_actual, 100, axis=0)
# qmin = np.percentile(pf_actual, 0, axis=0)
# target = np.array([qmax[0], qmax[1], qmin[2], qmin[3], qmin[4]])

print("pf actual max:", np.max(pf_actual, axis=0))
print("pf actual min:", np.min(pf_actual, axis=0))
print("pf actual median:", np.median(pf_actual, axis=0))

target = np.array([80,  55, 110, 100, 100]).astype(float)
dists = np.linalg.norm(pf_actual - target, axis=1)
real_aspi = pf_actual[dists.argmin()].astype(float)
print("pf solution closest to target:", real_aspi)

pf actual max: [130 117 146 137 139]
pf actual min: [32 30 40 41 41]
pf actual median: [ 83.  73. 105.  98.  92.]
pf solution closest to target: [ 78.  56. 109. 105. 102.]


[89. 81. 89. 96. 78.]

In [5]:
n_items = items.shape[0]
n_obj = items.shape[1] - 1
n_con = 1

if n_obj == 3:
    n_selected = 6
    max_row_diff = 5 
elif n_obj == 5:
    n_selected = 10
    max_row_diff = 500 
else:
    raise ValueError(f"Number of objectives {n_obj} not supported")

In [6]:
print("max:", np.max(items*n_selected, axis=0))
print("min:", np.min(items*n_selected, axis=0))
print("median:", np.median(items*n_selected, axis=0))

max: [170 190 180 170 170 190]
min: [10 10 10 10 20 10]
median: [ 80.  55. 110. 100. 100.  90.]


In [7]:
# biased eda run

capacity = n_selected * 10
pop_size = 1_000
generations = 100 
max_no_improve_gen = 5
temp = 0.1
aspi = np.array([80,  55, 110, 100, 100]).astype(float)

eda_process = eda_distance.KnapsackEDA(
    items=items,
    capacity=capacity,
    n_selected=n_selected,
    n_obj=n_obj,
    pop_size=pop_size,
    generations=generations,
    max_no_improve_gen=max_no_improve_gen,
    max_row_diff=max_row_diff,
    seed=1125,
    aspi=aspi,
    if_rank=True,
    temp=temp
)
results = eda_process.run()


result_type = "eda_human_reweight"
output_dir = "data/eda_results/test"
os.makedirs(output_dir, exist_ok=True)
file_path = os.path.join(output_dir, f"{result_type}_test6_3.pkl")
if os.path.exists(file_path):
    print(f"File {file_path} already exists")
else:
    with open(file_path, 'wb') as f:
        pickle.dump(results, f)

Mode 1 generation 1 (no improve count: 0)
Mode 1 generation 2 (no improve count: 0)
Mode 1 generation 3 (no improve count: 0)
Mode 1 generation 4 (no improve count: 0)
Mode 1 generation 5 (no improve count: 0)
Mode 1 generation 6 (no improve count: 0)
Mode 1 generation 7 (no improve count: 0)
Mode 1 generation 8 (no improve count: 0)
Mode 1 generation 9 (no improve count: 0)
Mode 1 generation 10 (no improve count: 0)
Mode 1 generation 11 (no improve count: 0)
Mode 1 generation 12 (no improve count: 1)
Mode 1 generation 13 (no improve count: 0)
Mode 1 generation 14 (no improve count: 0)
Mode 1 generation 15 (no improve count: 1)
Mode 1 generation 16 (no improve count: 2)
Mode 1 generation 17 (no improve count: 0)
Mode 1 generation 18 (no improve count: 0)
Mode 1 generation 19 (no improve count: 0)
Mode 1 generation 20 (no improve count: 0)
Mode 1 generation 21 (no improve count: 0)
Mode 1 generation 22 (no improve count: 0)
Mode 1 generation 23 (no improve count: 0)
Mode 1 generation 24

In [8]:
with open(f"data/eda_results/test/eda_human_reweight_test6_3.pkl", "rb") as f:
    results = pickle.load(f)
pf = results["converged_pf_table"][-1]

In [ ]:
objective_pairs = list(itertools.combinations(range(5), 2)) 
fig, axes = plt.subplots(2, 5, figsize=(24, 8))
axes = axes.ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    ax.plot(pf_actual[:, a], pf_actual[:, b], "go", alpha=0.2, markersize=3, label="PF")
    ax.plot(pf[:, a], pf[:, b], "bo", alpha=0.2, markersize=3, label="PF")
    # sns.kdeplot(
    #     x=pf_actual_norm[:, a],
    #     y=pf_actual_norm[:, b],
    #     levels=10,
    #     fill=True,   
    #     color="green",
    #     ax=ax
    # )
    # sns.kdeplot(
    #     x=pf_norm[:, a],
    #     y=pf_norm[:, b],
    #     levels=10,
    #     fill=True,   
    #     color="steelblue",
    #     alpha=0.4,
    #     ax=ax
    # )
    ax.plot(
        aspi[a], aspi[b],
        "rs", alpha=1, markersize=6, label="Aspiration"
    )
    ax.set_xlabel(f"Obj {a + 1}")
    ax.set_ylabel(f"Obj {b + 1}")
    ax.set_xlim(40, 140)
    ax.set_ylim(40, 140)
    
fig.tight_layout()
plt.show()

In [ ]:
def load_items(stimuli_dir: Path, trial_id: int) -> np.ndarray:
    csv_path = stimuli_dir / f"civ_items_trial_{trial_id}.csv"
    df = pd.read_csv(csv_path)
    numeric = df.drop(columns=["Name"], errors="ignore")
    return numeric.to_numpy(dtype=np.int64)

In [ ]:
def gen_aspi(ref_sol, items):
    n_obj = items.shape[1] - 1
    if n_obj == 3:
        n_selected = 6
    elif n_obj == 5:
        n_selected = 10
    else:
        raise ValueError(f"Number of objectives {n_obj} not supported")

    items_q5 = np.percentile(items[:, :n_obj], 5, axis=0)
    items_q95 = np.percentile(items[:, :n_obj], 95, axis=0)
    aspi = (ref_sol - items_q5*n_selected) / (items_q95*n_selected - items_q5*n_selected + 1e-12)
    return aspi 

In [ ]:
# load results
run_name = "test5"
trial_id = 8
stimuli_dir = Path("card_game/stimuli")
items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)

with open(f"data/eda_results/test/eda_human_reweight_{run_name}.pkl", "rb") as f:
    results = pickle.load(f)
pf = results["converged_pf_table"][-1]

with open(f"data/eda_results/test/history_{run_name}.pkl", "rb") as f:
    history = pickle.load(f)

with open(f'card_game/eda_results/eda_trial{trial_id}.pkl', 'rb') as f:
    results_actual = pickle.load(f)
pf_actual = results_actual['converged_pf_table'][-1]

In [ ]:
# obtain aspi for plotting
aspi = history["normalized_aspi"]

# normalize pf using min-max quantile
pf_norm = gen_aspi(pf, items)

# compute center solution then normalize it using min-max quantile
center = np.median(pf, axis=0)
dist = np.linalg.norm(pf - center, axis=1) 
center_sol = pf[dist.argmin()]
center_sol_norm = gen_aspi(center_sol, items)

# normalize pf_actual
pf_actual_norm = gen_aspi(pf_actual, items)

# plot normalized pf, normalized center, and aspiration 
objective_pairs = list(itertools.combinations(range(5), 2)) 
fig, axes = plt.subplots(2, 5, figsize=(24, 8))
axes = axes.ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    ax.plot(pf_actual_norm[:, a], pf_actual_norm[:, b], "go", alpha=0.2, markersize=3, label="PF")
    ax.plot(pf_norm[:, a], pf_norm[:, b], "bo", alpha=0.2, markersize=3, label="PF")
    # sns.kdeplot(
    #     x=pf_actual_norm[:, a],
    #     y=pf_actual_norm[:, b],
    #     levels=10,
    #     fill=True,   
    #     color="green",
    #     ax=ax
    # )
    # sns.kdeplot(
    #     x=pf_norm[:, a],
    #     y=pf_norm[:, b],
    #     levels=10,
    #     fill=True,   
    #     color="steelblue",
    #     alpha=0.4,
    #     ax=ax
    # )
    ax.plot(
        aspi[a], aspi[b],
        "rs", alpha=1, markersize=6, label="Aspiration"
    )
    ax.plot(
        center_sol_norm[a], center_sol_norm[b],
        "ks", alpha=1, markersize=6, label="Center"
    )
    ax.set_xlabel(f"Obj {a + 1}")
    ax.set_ylabel(f"Obj {b + 1}")
    ax.set_xlim(0.1, 0.9)
    ax.set_ylim(0.1, 0.9)
    
fig.tight_layout()
plt.show()

In [ ]:
import math

if "distribution_table" not in results:
    raise KeyError(
        "results has no 'distribution_table'. Re-run the EDA cell after updating "
        "eda_sol_reweight_noinit.py, or load a newer results pickle."
    )

# distribution_table / js_div_list start at Mode 1 gen 1 (no init).
# pareto_*_table[0] is generation 0 (initial population); Mode 1 starts at index 1.
distribution_table = np.vstack(results["distribution_table"])
mode1_gens = results["mode 1 generations"]
n_gens, n_items_dist = distribution_table.shape

item_names = pd.read_csv(stimuli_dir / f"civ_items_trial_{trial_id}.csv")["Name"].tolist()
x_items = np.arange(n_items_dist)


def _dist_phase(gen_num: int) -> str:
    """Absolute generation number (1 = first Mode 1 update)."""
    return "mode 1" if gen_num <= mode1_gens else "mode 2"


def plot_distribution_evolution(step: int = 1):
    """Overlay item-probability distributions across generations."""
    # dist index i <-> absolute generation i + 1
    gen_indices = list(range(0, n_gens, step))
    colors = plt.cm.viridis_r(np.linspace(0, 1, len(gen_indices)))

    fig, ax = plt.subplots(figsize=(12, 5))
    for color, dist_idx in zip(colors, gen_indices):
        gen_num = dist_idx + 1
        phase = _dist_phase(gen_num)
        ax.plot(
            x_items,
            distribution_table[dist_idx],
            color=color,
            alpha=0.7,
            linewidth=0.9,
            label=f"gen {gen_num} ({phase})" if dist_idx in (gen_indices[0], gen_indices[-1]) else None,
        )

    ax.set_xlabel("Item index")
    ax.set_ylabel("Probability")
    ax.set_title(f"Distribution evolution ({len(gen_indices)} generations)")
    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis_r, norm=plt.Normalize(1, n_gens))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.01)
    cbar.set_label("Generation")
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()


def plot_distribution_grid(step: int = 5, top_k: int = 10):
    """Bar charts of top-k items at selected generations."""
    gen_indices = list(range(0, n_gens, step))
    n_plots = len(gen_indices)
    cols = 5
    rows = math.ceil(n_plots / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    axes = np.atleast_1d(axes).ravel()

    for k, dist_idx in enumerate(gen_indices):
        ax = axes[k]
        gen_num = dist_idx + 1
        probs = distribution_table[dist_idx]
        top_idx = np.argsort(probs)[-top_k:][::-1]
        ax.bar(range(top_k), probs[top_idx], color="steelblue", alpha=0.85)
        ax.set_xticks(range(top_k))
        ax.set_xticklabels([item_names[i] for i in top_idx], rotation=60, ha="right", fontsize=7)
        ax.set_title(f"gen {gen_num} ({_dist_phase(gen_num)})")
        ax.set_ylabel("Probability")

    for ax in axes[n_plots:]:
        ax.axis("off")

    fig.suptitle(f"Top {top_k} items by probability", y=1.02)
    fig.tight_layout()
    plt.show()


def plot_js_divergence():
    """JS divergence across generations (starts at Mode 1 gen 1)."""
    js_div_list = results["js_div_list"]
    gens = np.arange(1, len(js_div_list) + 1)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(gens, js_div_list, "o-", markersize=3, linewidth=1)
    ax.axvline(x=mode1_gens + 0.5, color="red", linestyle="--", linewidth=1, label="mode 1 | mode 2")
    ax.set_xlabel("Generation")
    ax.set_ylabel("JS divergence")
    ax.set_title("Distribution convergence (JS divergence)")
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_distribution_evolution(step=1)
# plot_distribution_grid(step=max(1, n_gens // 10))
plot_js_divergence()


In [ ]:
import math

n_obj = items.shape[1] - 1
objective_pairs = list(itertools.combinations(range(n_obj), 2))
mode1_gens = results["mode 1 generations"]
# pareto_*_table[0] = generation 0 (initial population); Mode 1 starts at index 1
n_gens = len(results["pareto_front_table"])

# normalize reference pf_actual for background
pf_actual_norm = gen_aspi(pf_actual, items)
aspi_plot = history["normalized_aspi"]


def normalize_front(front: np.ndarray) -> np.ndarray:
    """Normalize objective values; handle pareto_front's extra constraint column."""
    obj = front[:, :n_obj] if front.shape[1] > n_obj else front
    return gen_aspi(obj, items)


def _pf_phase(gen_idx: int) -> str:
    if gen_idx == 0:
        return "init"
    if gen_idx <= mode1_gens:
        return "mode 1"
    return "mode 2"


def plot_generation_grid(
    obj_a: int,
    obj_b: int,
    *,
    step: int = 1,
    show_actual: bool = True,
):
    """Plot pareto_front_table and converged_pf_table for each generation."""
    gen_indices = list(range(0, n_gens, step))
    n_plots = len(gen_indices)
    cols = 5
    rows = math.ceil(n_plots / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    axes = np.atleast_1d(axes).ravel()

    for k, gen_idx in enumerate(gen_indices):
        ax = axes[k]

        if show_actual:
            ax.plot(
                pf_actual_norm[:, obj_a],
                pf_actual_norm[:, obj_b],
                "go",
                alpha=0.15,
                markersize=2,
                label="Actual PF" if k == 0 else None,
            )

        pf = normalize_front(results["pareto_front_table"][gen_idx])
        ax.plot(
            pf[:, obj_a],
            pf[:, obj_b],
            marker="o",
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor="k",
            markersize=3,
            alpha=0.7,
            label="Pareto front" if k == 0 else None,
        )

        # Mode 2 / converged_pf starts after init + all Mode 1 gens
        conv_idx = gen_idx - mode1_gens - 1
        if conv_idx >= 0:
            cpf = normalize_front(results["converged_pf_table"][conv_idx])
            ax.plot(
                cpf[:, obj_a],
                cpf[:, obj_b],
                "rs",
                alpha=0.25,
                markersize=2,
                label="Converged PF" if k == 0 else None,
            )

        ax.plot(
            aspi_plot[obj_a],
            aspi_plot[obj_b],
            "r*",
            markersize=8,
            label="Aspiration" if k == 0 else None,
        )

        ax.set_title(f"gen {gen_idx} ({_pf_phase(gen_idx)})")
        ax.set_xlabel(f"Obj {obj_a + 1}")
        ax.set_ylabel(f"Obj {obj_b + 1}")

    for ax in axes[n_plots:]:
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"Obj {obj_a + 1} vs Obj {obj_b + 1}", y=1.05)
    fig.tight_layout()
    plt.show()


# overview: obj 1 vs obj 2 for every generation
plot_generation_grid(0, 1, step=1)

# # --- results_actual (baseline EDA; no init slot at index 0) ---
# mode1_gens_actual = results_actual["mode 1 generations"]
# n_gens_actual = len(results_actual["pareto_front_table"])


# def _pf_phase_actual(gen_idx: int) -> str:
#     return "mode 1" if gen_idx < mode1_gens_actual else "mode 2"


# def plot_generation_grid_actual(
#     obj_a: int,
#     obj_b: int,
#     *,
#     step: int = 1,
# ):
#     """Plot pareto_front_table and converged_pf_table for results_actual."""
#     gen_indices = list(range(0, n_gens_actual, step))
#     n_plots = len(gen_indices)
#     cols = 5
#     rows = math.ceil(n_plots / cols)

#     fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
#     axes = np.atleast_1d(axes).ravel()

#     for k, gen_idx in enumerate(gen_indices):
#         ax = axes[k]

#         pf = normalize_front(results_actual["pareto_front_table"][gen_idx])
#         ax.plot(
#             pf[:, obj_a],
#             pf[:, obj_b],
#             marker="o",
#             linestyle="None",
#             markerfacecolor="none",
#             markeredgecolor="k",
#             markersize=3,
#             alpha=0.7,
#             label="Pareto front" if k == 0 else None,
#         )

#         conv_idx = gen_idx - mode1_gens_actual
#         if conv_idx >= 0:
#             cpf = normalize_front(results_actual["converged_pf_table"][conv_idx])
#             ax.plot(
#                 cpf[:, obj_a],
#                 cpf[:, obj_b],
#                 "rs",
#                 alpha=0.25,
#                 markersize=2,
#                 label="Converged PF" if k == 0 else None,
#             )

#         ax.plot(
#             aspi_plot[obj_a],
#             aspi_plot[obj_b],
#             "r*",
#             markersize=8,
#             label="Aspiration" if k == 0 else None,
#         )

#         ax.set_title(f"gen {gen_idx + 1} ({_pf_phase_actual(gen_idx)})")
#         ax.set_xlabel(f"Obj {obj_a + 1}")
#         ax.set_ylabel(f"Obj {obj_b + 1}")

#     for ax in axes[n_plots:]:
#         ax.axis("off")

#     handles, labels = axes[0].get_legend_handles_labels()
#     if handles:
#         fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))
#     fig.suptitle(f"results_actual: Obj {obj_a + 1} vs Obj {obj_b + 1}", y=1.05)
#     fig.tight_layout()
#     plt.show()


# plot_generation_grid_actual(0, 1, step=1)


In [ ]:
sns.heatmap(np.corrcoef(items.T), annot=True, cmap="coolwarm")